## 9. 权限系统与 Human-in-the-Loop

> 来源：[Configure permissions](https://code.claude.com/docs/en/agent-sdk/permissions)、[Handle approvals and user input](https://code.claude.com/docs/en/agent-sdk/user-input)


### 9.1 权限评估顺序（先记这条流水线）

Claude 每请求一次 tool，SDK 按固定顺序评估，**前面的步骤解决了，后面的就不再执行**：

```text
Hooks → Deny 规则 → Ask 规则 → Permission mode → Allow 规则 → can_use_tool 回调
```

每一步由哪个配置控制，先对上号。注意第 3 步：ask 规则在 `ClaudeAgentOptions` 里**没有对应字段**，只能写在 settings.json 的 `permissions.ask` 里：

| 步骤 | `ClaudeAgentOptions` 字段 | settings.json 键 |
|---|---|---|
| 1 Hooks | `hooks` | `hooks` |
| 2 Deny 规则 | `disallowed_tools` | `permissions.deny` |
| 3 Ask 规则 | —（无对应字段） | `permissions.ask` |
| 4 Permission mode | `permission_mode` | `permissions.defaultMode` |
| 5 Allow 规则 | `allowed_tools` | `permissions.allow` |
| 6 兜底回调 | `can_use_tool` | —（回调是代码，settings 写不了） |

1. **Hooks** 最先跑。hook 可以直接 deny；但 hook 返回 allow **不会跳过**后面的 deny/ask 规则。
2. **Deny 规则** 匹配即阻断，**连 `bypassPermissions` 都拦得住**。
3. **Ask 规则** 的语义是"这类调用必须先问过人"：匹配的调用直接落入 `can_use_tool` 回调等确认，allow 规则和 `bypassPermissions` 都豁免不了。"必须与人交互"的 tool 也总是落到回调，即便 allow 规则命中——这类 tool 有两个：`AskUserQuestion`，和 MCP server 用 `_meta["anthropic/requiresUserInteraction"]` 注解标记的自家工具（需 Claude Code ≥ v2.1.199，见 §12「外部 MCP」）——但 `dontAsk` 模式下这类交互工具不落回调、**直接被 deny**（该模式从不询问），headless agent 里用不了 `AskUserQuestion`。
4. **Permission mode**：`bypassPermissions` 批准到这一步的一切；`acceptEdits` 批准文件操作；`plan` 把写操作强制引回回调（无视 allow 规则）。
5. **Allow 规则** 匹配即批准。
6. **`can_use_tool` 回调** 兜底；`dontAsk` 模式下跳过此步直接 deny。

拿一份具体配置走一遍：options 里配第 2、4、5、6 步，第 3 步的 ask 规则放在项目的 `.claude/settings.json` 里。示例中的 `gate` 不是 SDK 的名字，是应用自己写的一个 async 审批回调函数——SDK 走到第 6 步（或第 3 步 ask 命中）时带着 tool 名和入参调用它，由它返回放行或拒绝；签名与实现见 §9.3「can_use_tool 回调」，此处先当"负责点头的门卫"。

```python
options = ClaudeAgentOptions(
    disallowed_tools=["Bash(rm *)"],   # 第 2 步：deny 规则（带范围）
    permission_mode="default",         # 第 4 步：模式
    allowed_tools=["Read"],            # 第 5 步：allow 规则
    can_use_tool=gate,                 # 第 6 步：兜底回调（gate 的定义见 §9.3）
)
```

```json
{
  "permissions": {
    "ask": ["Bash(git push:*)"]
  }
}
```

这份配置下，四次 tool 请求各停在流水线的哪一步：

```python
# Bash("rm -rf build")     → 第 2 步 deny 命中，直接阻断；gate 不会被调用
# Bash("git push origin")  → 第 3 步 ask 命中，强制交给 gate 问人（加 allow 规则也豁免不了）
# Read("notes.md")         → 第 5 步 allow 命中，自动批准；gate 不会被调用
# Bash("ls")               → 前五步都没接住，落到第 6 步，gate 说了算
```

由这条流水线推出四个关键结论：

- **被前面任何一步批准的调用，永远到不了 `can_use_tool`**——上例里 `Read` 根本不经过 gate，写在回调里的检查对 `allowed_tools` 里的 tool 静默失效。覆盖面取决于写法：裸名 `"Read"` 自动批准该 tool 的全部调用；allow 规则同样可以带范围——`"Bash(ls *)"` 只自动批准匹配的调用，其余 `Bash` 调用仍落回调（例外是第 3 步那两类交互工具，allow 命中也照样进回调）。必须每次调用都生效的逻辑，用 `PreToolUse` hook（它在整条流水线之前，连 `bypassPermissions` 都绕不过它的 deny）。
- **`allowed_tools` 不约束 `bypassPermissions`**：`allowed_tools=["Read"]` + `bypassPermissions` 仍然批准所有工具——所有调用在第 4 步就被放行，第 5 步的名单形同虚设。要在该模式下禁工具只能用 `disallowed_tools`（第 2 步，在模式之前）。
- **deny 规则两种写法语义不同**：裸名 `"Bash"` 把 tool 定义整个从 context 移除（Claude 根本看不见）；带范围 `"Bash(rm *)"` 保留 tool、只 deny 匹配的调用（上例里 `Bash("ls")` 照常往下走）。
- **通配符两侧不对称**：deny 侧任意通配——`disallowed_tools=["*"]` 匹配全部工具（定义整个移除），`"mcp__*"` 匹配所有 server 的全部 MCP 工具；allow 侧必须点名到具体 server——通配符只能跟在写死的 server 名后面，`"mcp__github__*"`（github server 的全部工具）、`"mcp__github__get_*"`（其中 `get_` 开头的）都合法，而 `allowed_tools=["*"]` 和 `["mcp__*"]` 这种 server 名也想通配的写法**不报错但也不生效**：启动时告警一条、该规则被忽略，什么都不会批准。不对称的来由：allow 是自动批准，规则必须落到一个明确配置过的 server 上；deny 是拦截，范围写大无妨。

一个生效前提：settings.json 里的 allow/deny/ask 规则要参与评估，`setting_sources` 必须含 `"project"`（省略时默认含；显式传了列表就得自己带上，见 §20.1「三个 source 的语义」）——否则这些规则**静默失效**。ask 规则只有这一条配置路径，所以这个前提对它是硬依赖。

### 9.2 permission_mode 全表

先立定位：`permission_mode` 是**一个全局参数，§9.1 的流水线上有两处读它**——**第 4 步**读它决定"自动批准哪些调用"；**第 6 步**也读它决定"走到兜底这步的调用是进 `can_use_tool` 回调询问，还是直接 deny"（只有 `dontAsk` 选后者）。所以每个模式的行为归结为两个问题：**第 4 步自动批准什么？走到第 6 步的调用是问还是拒？**

| 模式 | 第 4 步自动批准什么 | 走到第 6 步的调用是问还是拒 | 适用 |
|---|---|---|---|
| `default` | 什么都不批，调用原样走向第 5 步 | 问：进 `can_use_tool` 回调；没配回调则 deny | 交互应用 + 审批回调 |
| `acceptEdits` | 文件编辑与文件系统命令（`mkdir`/`touch`/`rm`/`rmdir`/`mv`/`cp`/`sed`），仅限 `cwd` 和 `add_dirs` 范围内；写受保护路径不批 | 问，同 `default` | 受信任的开发工作流 |
| `plan` | 什么都不批，还反向收紧：写操作**强制**送回调，`Edit` 在 allow 名单里也无效；只读工具照常 | 问 | 先规划再动手 |
| `dontAsk` | 什么都不批 | **拒：直接 deny，回调从不被调用**；`AskUserQuestion` 等交互工具同样直接被 deny | 锁死的 headless agent |
| `bypassPermissions` | **批准走到这一步的一切**——hooks/deny/ask 是第 1–3 步、在它前面，仍拦得住；Unix root 下不可用 | ——调用到不了第 6 步 | 沙箱 CI、隔离环境 |

（TypeScript 另有模型分类器审批的 `auto` 模式，Python 无。）

推荐配对（模式 × allow 名单 × 回调的组合）：

- 交互应用：`default` + `can_use_tool` 回调——常规操作走 allow 名单自动批，名单外的弹给真人。
- 开发机自主 agent：`acceptEdits`——改文件不打断，其余照常审。
- 锁死的 headless agent：`allowed_tools` + `dontAsk`——名单即全集，名单外硬拒，不依赖"恰好没配回调"这种偶然。
- `bypassPermissions` 只留给容器/CI 等隔离环境。

**MCP tool 的授权用 `allowed_tools` 通配符（如 `mcp__github__*`），不要靠放宽 permission mode**——`acceptEdits` 只批文件类操作、批不到 MCP tool，`bypassPermissions` 又什么都批、放得太宽。

`acceptEdits` 批不到 MCP tool 的底层原因：权限系统只按**工具名和入参**裁决"这次调用放不放行"，不看也看不见工具内部做什么，而 `acceptEdits` 批准的是一份写死的名单（`Edit`/`Write`/`NotebookEdit` 加上表那几个文件系统命令）。由此推出两面：一个会写文件的 MCP tool，不因"写文件"这个行为被 `acceptEdits` 批准；反过来它一旦被放行，handler 内部的文件改动就再没有任何关卡——不受 `cwd` 范围约束，file checkpointing 也不跟踪（它只跟踪 `Write`/`Edit`/`NotebookEdit`，§21.4「文件快照与任务清单」）。要管住这类工具，在 handler 里自行校验，或用 `PreToolUse` hook 检查入参。

> [!warning] 子 agent 继承父级模式且不可覆盖
> 父级用 `bypassPermissions` 或 `acceptEdits` 时，全部子 agent 强制继承同一模式，子 agent 自己的 `AgentDefinition.permissionMode` 字段写了也无效（§14.1「编程式定义」）。危险主要在 `bypassPermissions`：继承它的子 agent 获得无审批的完全系统访问，而子 agent 的 system prompt 是单独写的，约束往往比主 agent 松。这种情况下唯一还拦得住的是 ask 规则（settings.json 的 `permissions.ask`）——它在流水线第 3 步、排在模式之前，命中的调用仍会强制送回调询问（§9.1「权限评估顺序」）。

运行期可切模式：`ClaudeSDKClient` 上调 `await client.set_permission_mode("acceptEdits")`。典型用法是先严后松——先用 `default` 盯着 agent 走头几步，确认方向没错，再切到 `acceptEdits` 放手让它改。

### 9.3 `can_use_tool` 回调（HITL 的落点）

先分清谁在哪儿裁决：§9.1 的整条流水线跑在 **CLI 子进程内部**，但六步吃的配置分两类。deny/allow 名单、ask 规则、permission_mode 是**配置**，启动时一次性下发给 CLI，之后每次 tool 调用 CLI 就地裁决，不与 SDK 往来；`can_use_tool` 和 Python hook 是**代码**，函数体活在应用进程里，CLI 执行不了——走到这两处时，CLI 从 stdout 发一条询问并挂起，SDK 调完函数把结果写回 CLI 的 stdin，CLI 才继续。这就是"配置能下发，代码只能回调"；下方 warning 里 stdin 的坑只坑回调不坑名单，原因也在此——名单裁决不过 stdio，回调必须过。

回调签名与返回值：

```python
async def can_use_tool(
    tool_name: str,                    # "Bash" / "Write" / "AskUserQuestion"...
    input_data: dict,                  # tool 入参，内容随 tool 而异
    context: ToolPermissionContext,    # 上下文，字段见下
) -> PermissionResultAllow | PermissionResultDeny: ...
```

`ToolPermissionContext` 的字段：`suggestions`（CLI 给出的现成权限更新建议）、`blocked_path`（触发询问的文件路径，比如 Bash 访问了允许目录之外的路径）、`decision_reason`（`PreToolUse` hook 返回 `ask` 时转发过来的理由）、`title` / `display_name` / `description`（权限 UI 的现成文案：完整提示句 / 按钮短语 / 副标题）、`signal`（保留位，当前未启用）。

- `PermissionResultAllow(updated_input=...)`：放行，还能**改写工具入参**（比如把写路径重定向到沙箱；Claude 不知道被改过）。
- `PermissionResultAllow(updated_input=..., updated_permissions=...)`：批准并持久化规则（从 `context.suggestions` 里挑 `destination == "localSettings"` 的条目回传，即"总是允许"；需 SDK ≥ 0.1.80）。`updated_permissions` 收的是 `PermissionUpdate` 列表，能表达的不止"加一条 allow 规则"：`type` 有六种操作——`addRules` / `replaceRules` / `removeRules` / `setMode` / `addDirectories` / `removeDirectories`；`destination` 有四档——`userSettings` / `projectSettings` / `localSettings` / `session`（session 档只在本会话生效、不落盘）。挑 suggestions 里的 localSettings 回传只是其中最常用的一种。
- `PermissionResultDeny(message=..., interrupt=...)`：拒绝并给模型一个理由——理由写得好，Claude 会换方案（如"用户不想删文件，问能否改成压缩归档"）。
- 还有一种**整体改道**：不批也不驳这次调用，直接用流式输入给 Claude 发一条全新指令接管方向（§16「流式输入」）。

因为回调是 `async`，可以在里面 `await` 一个 Future 挂起，等真人点"批准/拒绝"再返回——把"异步等人"伪装成一次同步权限判断，Human-in-the-Loop 的本质就是这个。回调可以无限期挂起等待；但要等的时间一旦超过进程本身的存活时间，就不能硬等——改用 hook 的 `defer` 决策先让进程退出，之后再 resume 接着处理（§11.3「回调签名与返回值」）。

> [!warning] Python 专属坑：can_use_tool 的两条生效路线
> 
> 结论先行：Python 里要用 `can_use_tool`，优先直接用 `ClaudeSDKClient`，零额外动作；坚持用 `query()` 就必须**同时**做两件事——`prompt` 传 async generator（不能传字符串）+ 注册一个 dummy `PreToolUse` hook。这两件不是二选一，缺任何一件的表现都一样：不报错，回调永远不触发。
>
> 为什么，拆成三句：
>
> 1. **审批的问与答走 SDK 与 CLI 子进程之间的 stdio**：CLI 从 stdout 发出询问"这个 tool 能不能用"，应用的答复必须写回 CLI 的 stdin。
> 2. **询问来得晚**：CLI 里的模型要先读 prompt、思考、决定调工具，之后才发审批询问。
> 3. **SDK 关 stdin 的时机比询问更早**：字符串 prompt 发完即关输入端，generator 的消息发完也关（EOF＝输入结束）。stdin 一关，答复无路可走，审批永远等不到回音。
>
> 四种用法就是四种 stdin 命运：
>
> | 用法 | stdin 何时关 | 结果 |
> |---|---|---|
> | `query(prompt="字符串")` | prompt 一次性交给 CLI，输入端随即关闭 | 询问到达时无处写答复，回调不生效 |
> | `query(prompt=生成器)` | 生成器消息发完，SDK 关 stdin | 询问多半在这之后才来，照样落空 |
> | `query(prompt=生成器)` + dummy `PreToolUse` hook | SDK 知道随时可能要写 hook 答复，**发完也不关** | ✅ 官方推荐写法，即下方示例 |
> | `ClaudeSDKClient` | 为支持多轮追问，stdin 全程不关 | ✅ 天然没这个问题，无需 dummy hook |
>
> dummy hook 是一个**注册了但什么都不干的"摆设钩子"**：
>
> ```python
> async def dummy_hook(input_data, tool_use_id, context):
>     return {"continue_": True}   # 一律放行、不拦不改；键名带下划线是避开 Python 关键字 continue
> ```
>
> 注册它不是为了它做什么，而是为了它**存在**：SDK 看到有 hook 注册，就认为随时可能要往 stdin 写 hook 答复，于是 generator 发完也不关 stdin——审批询问因此有路可答。所以"流式输入"和"dummy hook"两个前提是同一句话说两遍：**别让 stdin 提前关**。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
from claude_agent_sdk.types import (
    HookMatcher,
    PermissionResultAllow,
    PermissionResultDeny,
    ToolPermissionContext,
)


async def gate(tool_name: str, input_data: dict, context: ToolPermissionContext):
    # 例1：禁止危险删除，并给模型一个可行的替代方向
    if tool_name == "Bash" and "rm " in input_data.get("command", ""):
        return PermissionResultDeny(
            message="User doesn't want to delete files; compress them into an archive instead."
        )
    # 例2：把碰 config 的写操作重定向到沙箱（改写入参后放行，Claude 无感知）
    if tool_name in ("Write", "Edit") and "config" in input_data.get("file_path", ""):
        safe = f"./sandbox/{input_data['file_path']}"
        return PermissionResultAllow(updated_input={**input_data, "file_path": safe})
    # 例3（HITL 骨架）：真人审批
    #   fut = asyncio.get_running_loop().create_future()
    #   push_to_ui(tool_name, input_data, fut)   # 前端弹审批框，点击后 fut.set_result(bool)
    #   return PermissionResultAllow(updated_input=input_data) if await fut \
    #       else PermissionResultDeny(message="user denied")
    return PermissionResultAllow(updated_input=input_data)


# Python 必需的 workaround：dummy PreToolUse hook 保持流打开，否则回调不会被触发
async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}


# can_use_tool 要求流式输入：prompt 传 async generator 而非字符串
async def prompt_stream():
    yield {
        "type": "user",
        "message": {"role": "user", "content": "Update the app config file"},
    }


async def demo_gate():
    async for m in query(
        prompt=prompt_stream(),
        options=ClaudeAgentOptions(
            can_use_tool=gate,
            hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[dummy_hook])]},
        ),
    ):
        if isinstance(m, ResultMessage) and m.subtype == "success":
            print(m.result)


await demo_gate()

### 9.4 另一条 HITL 通道：`AskUserQuestion`

`can_use_tool` 解决的是"Claude 想做某事，人来批"；`AskUserQuestion` 方向相反——**Claude 拿不准方向时，主动出题请人选**。它是个内建工具（`plan` 模式下尤其常用），在 Claude 眼里就是一次普通的 tool 调用：把问题和选项放进工具入参发出去，从 tool result 里读回答案。

但 SDK 是个无界面的库，不会自己弹窗——"把问题呈现给真人、收回答案"必须由应用完成。应用做这件事的挂载点还是 `can_use_tool` 回调，整个问答是**一次借用审批通道完成的往返**：

```text
① Claude 调用 AskUserQuestion，questions 在工具入参里
② 交互类工具的调用一律落入 can_use_tool 回调（§9.1「权限评估顺序」第 3 步，allow 名单也豁免不了）
③ 应用在回调里认出 tool_name == "AskUserQuestion"，把 questions 呈现给真人、收集选择
④ 回调返回 PermissionResultAllow(updated_input={questions 原样, answers})——答案藏在改写后的入参里
⑤ 工具带着 answers "执行"，Claude 从 tool result 里读到答案
```

问题借工具入参进来，答案借 `updated_input` 回去——§9.3「can_use_tool 回调」里"改写入参后放行"的技巧，在这里是唯一正道。注意问题和选项全部由 Claude 生成，应用不能往这个流程里塞自己的问题；应用想主动问用户，那是应用自身的 UI 逻辑，与这个工具无关。

**输入格式**（第 ① 步，`input_data["questions"]`，每次 1–4 个问题）：

| 字段 | 含义 |
|---|---|
| `question` | 完整问题文本 |
| `header` | 短标签（≤12 字符） |
| `options` | 2–4 个选项，各有 `label` 和 `description` |
| `multiSelect` | `true` 则可多选 |

**回填格式**（第 ④ 步的具体写法）：`updated_input` 必须包含**原样传回的 `questions`**，再加一个 `answers` dict——key 是问题文本，value 是所选 `label`（多选传 label 列表）。用户打了自由文本就直接放原文，不要放 "Other"：

```python
return PermissionResultAllow(
    updated_input={
        "questions": input_data.get("questions", []),
        "answers": {
            "How should I format the output?": "Summary",
            "Which sections should I include?": ["Introduction", "Conclusion"],
        },
    }
)
```

用户不按题回答、整体打了一段话时，放进与 `questions` 并列的顶层 `response` 字段——Claude 收到的是 "The user responded: …" 而非逐题答案。

三个限制：子 agent 内当前不可用 `AskUserQuestion`；`dontAsk` 模式下它直接被 deny（§9.1「权限评估顺序」第 3 步）；如果用 `tools` 字段收窄了工具集，必须把 `"AskUserQuestion"` 加回名单，否则 Claude 无法提问。

两条通道最终都汇成"agent 停下等人输入"的状态。更复杂的交互（表单、多步向导、对接外部审批系统）用自定义工具实现（§10「自定义工具」），那是控制力最强、实现成本也最高的一档。完整的终端问答实现（展示问题 → 收输入 → 数字选项或自由文本解析 → 回填）见下方 cell。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions
from claude_agent_sdk.types import HookMatcher, PermissionResultAllow


def ask_user(questions: list) -> dict:
    """把 Claude 的问题呈现给真人：输入数字选选项，直接打字算自由回答。"""
    answers = {}
    for q in questions:
        print(f"\n[{q.get('header', '')}] {q['question']}")
        options = q.get("options", [])
        for i, opt in enumerate(options, 1):
            print(f"  {i}. {opt['label']} —— {opt.get('description', '')}")
        raw = input("> ").strip()
        if raw.isdigit() and 1 <= int(raw) <= len(options):
            answers[q["question"]] = options[int(raw) - 1]["label"]
        else:
            answers[q["question"]] = raw  # 自由文本原样放入，不要写 "Other"
    return answers


async def gate(tool_name, input_data, context):
    if tool_name == "AskUserQuestion":
        return PermissionResultAllow(
            updated_input={
                "questions": input_data.get("questions", []),  # 原样传回
                "answers": ask_user(input_data.get("questions", [])),
            }
        )
    return PermissionResultAllow(updated_input=input_data)


async def dummy_hook(input_data, tool_use_id, context):
    return {
        "continue_": True
    }  # §9.3「can_use_tool 回调」 的 Python workaround：保持流打开


async def prompt_stream():
    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": "Plan a refactor of the auth module; ask me before choosing a direction.",
        },
    }


async def demo_ask_user_question():
    async for m in query(
        prompt=prompt_stream(),
        options=ClaudeAgentOptions(
            permission_mode="plan",  # plan 模式下 Claude 更常主动提问
            can_use_tool=gate,
            hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[dummy_hook])]},
        ),
    ):
        if hasattr(m, "result"):
            print(m.result)


await demo_ask_user_question()